# Lab 3 — Send a city IoT reading to Zerobus

You'll push one row directly into the Delta table `ops_data.zerobus.measurements`
via the **official Zerobus Ingest SDK** (gRPC under the hood). The SDK handles
OAuth, `authorization_details`, and stream lifecycle — you only edit the three
widgets at the top: city, temperature, and an optional comment.

Credentials (service principal client_id / secret, Zerobus endpoint, workspace URL)
are read at runtime from the shared UC config table `ops_data.zerobus.config`,
populated by the setup notebook. You never paste them.

Schema of the target table:

| column      | type                                    |
|-------------|-----------------------------------------|
| id          | STRING (UUID, generated per submission) |
| city        | STRING                                  |
| temperature | FLOAT                                   |
| comment     | STRING (optional free-form note)        |

## Install the Zerobus Ingest SDK

Run this cell first. It installs the SDK and restarts Python so the install takes
effect (~10-30 seconds). Then run the rest of the cells in order.

In [0]:
%pip install --quiet "databricks-zerobus-ingest-sdk>=1.0.0"
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## ✏️ Edit this cell — your input

Change the widget values at the top of the notebook (City, Temperature °C, Comment),
then run the **Submit** cell below. Re-run as many times as you like; each run
writes one new row with a fresh UUID. Comment may be left blank.

In [0]:
dbutils.widgets.text("city",        "Munich",        "City")
dbutils.widgets.text("temperature", "21.5",          "Temperature (°C)")
dbutils.widgets.text("comment",     "Hello Zerobus", "Comment (optional)")

CITY        = dbutils.widgets.get("city").strip()
TEMPERATURE = float(dbutils.widgets.get("temperature"))
COMMENT     = dbutils.widgets.get("comment")

assert CITY, "Enter a city."
print(f"Prepared record: city={CITY!r}  temperature={TEMPERATURE}  comment={COMMENT!r}")

Prepared record: city='Munich'  temperature=21.5  comment='Hello Zerobus'


## ⛔ DO NOT MODIFY — Zerobus SDK client (plumbing)

Reads credentials from `ops_data.zerobus.config`, opens a stream via
`ZerobusSdk.create_stream(...)`, ingests one JSON record, flushes, and closes.
The SDK handles OAuth and `authorization_details` for us. If anything here breaks,
flag your instructor — don't edit.

In [0]:
import json
import uuid

from zerobus.sdk.sync import ZerobusSdk
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties

CATALOG = "ops_data"
SCHEMA  = "zerobus"
TABLE   = "measurements"

_CONFIG        = spark.table(f"{CATALOG}.{SCHEMA}.config").first()
_CLIENT_ID     = _CONFIG["client_id"]
_CLIENT_SECRET = _CONFIG["client_secret"]
_WORKSPACE_URL = _CONFIG["workspace_url"]
_WORKSPACE_ID  = _CONFIG["workspace_id"]
# The SDK wants the bare host (no scheme, no path) for the gRPC endpoint:
_SERVER_ENDPOINT = _CONFIG["zerobus_endpoint"].replace("https://", "").rstrip("/")
print(f"Config loaded for workspace_id={_WORKSPACE_ID}, endpoint={_SERVER_ENDPOINT}")


def submit_iot_record(city: str, temperature: float, comment: str = "") -> dict:
    """Send one {id, city, temperature, comment} record via the Zerobus Ingest SDK."""
    record = {
        "id":          str(uuid.uuid4()),
        "city":        city,
        "temperature": float(temperature),
        "comment":     comment or "",
    }
    sdk = ZerobusSdk(_SERVER_ENDPOINT, unity_catalog_url=_WORKSPACE_URL)
    table_props = TableProperties(f"{CATALOG}.{SCHEMA}.{TABLE}")
    options     = StreamConfigurationOptions(record_type=RecordType.JSON)
    stream      = sdk.create_stream(_CLIENT_ID, _CLIENT_SECRET, table_props, options)
    try:
        stream.ingest_record(json.dumps(record))
        stream.flush()
    finally:
        stream.close()
    return record

Config loaded for workspace_id=7474659435544366, endpoint=7474659435544366.zerobus.us-west-2.cloud.databricks.com


## 📤 Submit — run to send your reading

In [0]:
sent = submit_iot_record(CITY, TEMPERATURE, COMMENT)
print(f"✅ Sent to {CATALOG}.{SCHEMA}.{TABLE}: {sent}")

2026-06-12T20:43:17.083184Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=670d5948-33a3-40c2-a28a-e70e506bf415
2026-06-12T20:43:17.083227Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=670d5948-33a3-40c2-a28a-e70e506bf415
2026-06-12T20:43:17.287552Z  INFO databricks_zerobus_ingest_sdk: Closing stream stream_id=670d5948-33a3-40c2-a28a-e70e506bf415
✅ Sent to ops_data.zerobus.measurements: {'id': '6a29cc72-ffe1-4f0a-9e40-1a3f963da5cb', 'city': 'Munich', 'temperature': 21.5, 'comment': 'Hello Zerobus'}


## 🔍 Verify — your row should appear (may take a few seconds)

In [0]:
%sql
SELECT id, city, temperature, comment
FROM ops_data.zerobus.measurements
ORDER BY city, temperature DESC;

id,city,temperature,comment
668acf1f-267f-41bd-8d6d-ac7836b4c8ad,Atlanta,72.5,Go Bravbes :)
489c1d6e-f430-4ab1-9a48-cf3c2b602011,Atlanta,72.5,Go Bravbes :)
0e7c5839-0584-4161-9355-edf64907ff1b,Atlanta,72.5,Go Braves :)
f19598b3-782b-4696-b5a5-e40daa2273e8,Bangalore,17.5,Hello Bangalore Zero
96f36887-6573-49cf-a1a9-82c12a73ad23,Calagry,21.5,Hello Zerobus
fbe1e123-05fb-4ed2-ac06-19ba7950bc79,Calgary,9.0,Hello Summer
c5f33781-250f-4274-a861-2cbd08b4a9b5,Chennai,30.0,Hello!
af06322d-5cca-41b4-bc19-430efadedc29,Chicago,22.0,Hello Zerobus
4ae17ad4-b3fc-4ac1-96ac-688365deebba,Columbus,14.0,Hello Columbus
0af554b7-9f38-4969-8a84-4f94475408d4,El Paso,23.5,Hello Ash
